# Silver Layer: Product Info CDC

**Source**: `workspace.bronze.crm_prd_info_cdc`  
**Target**: `workspace.silver.crm_products_cdc`  
**Primary Key**: `prd_id` → `product_id`

**Transformations:**
1. Trim strings
2. Parse product key to extract category_id
3. Cost cleanup (replace nulls with 0)
4. Normalize product line
5. Cast dates
6. Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length, substring, regexp_replace, coalesce, lit, upper
from delta.tables import DeltaTable

# Table names
bronze_table = "workspace.bronze.crm_prd_info_cdc"
silver_table = "workspace.silver.crm_products_cdc"

print("✓ Configuration loaded")
print(f"  Source: {bronze_table}")
print(f"  Target: {silver_table}")

In [0]:
# Check if Silver exists and get watermark
silver_exists = spark.catalog.tableExists(silver_table)

if silver_exists:
    watermark = spark.sql(f"""
        SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01') as max_timestamp
        FROM {silver_table}
    """).collect()[0]["max_timestamp"]
    print(f"✓ Silver exists - Watermark: {watermark}")
else:
    watermark = "1900-01-01"
    print(f"ℹ️  Initial load - Watermark: {watermark}")

In [0]:
# Read NEW records from Bronze CDC
df = spark.table(bronze_table).filter(col("ingestion_timestamp") > watermark)

new_count = df.count()
print(f"📊 New records: {new_count:,}")

if new_count == 0:
    print("⏭️  No new data - exiting")
    dbutils.notebook.exit("No new data")

In [0]:
# 1. Trim strings
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

print("✓ Trimmed strings")

In [0]:
# 2. Parse product key to extract category_id
df = df.withColumn("cat_id", 
                    regexp_replace(substring(col("prd_key"), 1, 5), "-", "_"))

df = df.withColumn("prd_key", 
                   substring(col("prd_key"), 7, length(col("prd_key"))))

print("✓ Parsed product key")

In [0]:
# 3. Cost cleanup - replace nulls with 0
df = df.withColumn("prd_cost", coalesce(col("prd_cost"), lit(0)))

print("✓ Cleaned product cost")

In [0]:
# 4. Normalize product line
df = (
    df.withColumn(
        "prd_line",
        F.when(upper(col("prd_line")) == "M", "Mountain")
         .when(upper(col("prd_line")) == "R", "Road")
         .when(upper(col("prd_line")) == "S", "Other Sales")
         .when(upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

print("✓ Normalized product line")

In [0]:
# 5. Cast start date
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

print("✓ Cast dates")

In [0]:
# 6. Rename columns
rename_map = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

print("✓ Renamed columns")
df.limit(5).display()

In [0]:
if not silver_exists:
    print("Creating Silver table...")
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table))
    print(f"✓ Created: {spark.table(silver_table).count():,} records")
else:
    print("Merging into Silver...")
    target = DeltaTable.forName(spark, silver_table)
    (target.alias("target")
      .merge(df.alias("source"), "target.product_id = source.product_id")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())
    print(f"✓ MERGE complete: {spark.table(silver_table).count():,} total records")